# Shakespeare's Characters

The goal of this notebook is to train a simple LSTM model **to mimic the dialogue of Shakespearean characters.** We will train the model on a large corpus of text from Shakespeare's plays, teaching it **to predict the next character in a sequence.** For example, given the input 'to be or no', the model should predict 't' as the most likely following character. The input window then slides forward by one position—now reading 'o be or not'—to predict the next token. This sliding window approach allows the model to generate text character-by-character continuously.

In [5]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np

In [6]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

## Data Loading

In [7]:
# Download and prepare data
from pathlib import Path
import urllib.request

input_path = Path.cwd() / "input.txt"
if not input_path.exists():
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    urllib.request.urlretrieve(url, input_path)

text = input_path.read_text(encoding="utf-8")

In [8]:
# Create character-level vocabulary
chars = sorted(list(set(text)))

vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

# Encode the entire text into integers
data = [stoi[c] for c in text]

In [9]:
# Sliding Window Dataset
# Instead of sentences, we slice the giant text into overlapping sequences

class CharDataset(Dataset):
    def __init__(self, data, seq_len=100):
        self.data = data
        self.seq_len = seq_len

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, idx):
        # x is a sequence of seq_len characters
        # y is the single character that comes right after it
        x = self.data[idx:idx+self.seq_len]
        y = self.data[idx+self.seq_len]
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)

In [10]:
# Parameters
seq_len = 100
batch_size = 128

# Create the dataset
dataset = CharDataset(data, seq_len)

# Create the data loader
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

## Model Definition

In [11]:
class LSTM(nn.Module):
    def __init__(self, vocab_size, embed_size=64, hidden_size=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        emb = self.embedding(x)
        out, _ = self.lstm(emb)
        last_hidden = out[:, -1, :]
        logits = self.fc(last_hidden)
        return logits

In [12]:
# Instantiate the model
model = LSTM(vocab_size).to(device)

## Training

In [13]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.002)

In [14]:
epochs = 5
model.train()

for epoch in range(epochs):
    total_loss = 0
    for i, (x, y) in enumerate(loader):
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()

        logits = model(x)

        loss = criterion(logits, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if i % 500 == 0:
            print(f"Epoch {epoch+1}, Batch {i}, Loss: {loss.item():.4f}")

    print(f"--- Epoch {epoch+1} Avg Loss: {total_loss/len(loader):.4f} ---")

Epoch 1, Batch 0, Loss: 4.1697
Epoch 1, Batch 500, Loss: 2.0396
Epoch 1, Batch 1000, Loss: 1.6796
Epoch 1, Batch 1500, Loss: 2.0254
Epoch 1, Batch 2000, Loss: 2.0518
Epoch 1, Batch 2500, Loss: 1.5727
Epoch 1, Batch 3000, Loss: 1.6081
Epoch 1, Batch 3500, Loss: 1.7429
Epoch 1, Batch 4000, Loss: 1.7567
Epoch 1, Batch 4500, Loss: 1.3811
Epoch 1, Batch 5000, Loss: 1.6638
Epoch 1, Batch 5500, Loss: 1.6474
Epoch 1, Batch 6000, Loss: 1.5539
Epoch 1, Batch 6500, Loss: 1.6882
Epoch 1, Batch 7000, Loss: 1.8100
Epoch 1, Batch 7500, Loss: 1.3062
Epoch 1, Batch 8000, Loss: 1.5687
Epoch 1, Batch 8500, Loss: 1.4265
--- Epoch 1 Avg Loss: 1.7258 ---
Epoch 2, Batch 0, Loss: 1.4967
Epoch 2, Batch 500, Loss: 1.4191
Epoch 2, Batch 1000, Loss: 1.3940
Epoch 2, Batch 1500, Loss: 1.5481
Epoch 2, Batch 2000, Loss: 1.6140
Epoch 2, Batch 2500, Loss: 1.6310
Epoch 2, Batch 3000, Loss: 1.6679
Epoch 2, Batch 3500, Loss: 1.4187
Epoch 2, Batch 4000, Loss: 1.3568
Epoch 2, Batch 4500, Loss: 1.5472
Epoch 2, Batch 5000, Lo

## Validation

In [19]:
def generate_text(model, start_str="ROMEO: ", length=200, temperature=0.5):
    model.eval()
    generated = start_str

    for _ in range(length):
        # Format input string to length seq_len
        input_tokens = [stoi[c] for c in generated[-seq_len:]]
        x = torch.tensor(input_tokens).unsqueeze(0).to(device)

        with torch.no_grad():
            logits = model(x)

            # Apply temperature and sample (instead of argmax)
            # Higher temp = more "creative"/chaotic text
            probs = torch.softmax(logits / temperature, dim=-1)
            pred_idx = torch.multinomial(probs, num_samples=1).item()

            generated += itos[pred_idx]

    return generated

In [20]:
print("\nGenerated Shakespeare:\n")
print(generate_text(model, start_str="KING: ", length=300))


Generated Shakespeare:

KING: is the bold,
And like a man made her enemy to make
The eyes the death to the company of married
Unto the damparcy of the marriage,
I will be to the bear to the state and brother,
Which with the comment to the common head.

QUEEN ELIZABETH:
I would the suitious Bonienting himself,
To be the stars to 
